# Main

In [1]:
import os
os.chdir("../")

In [2]:
os.getcwd()

'/Users/supawitjunsiritrakhoon/Desktop/Customer_Churn_Prediction/Project_file/customer-churn-prediction'

In [3]:
from typing import Optional
from pathlib import Path
from datetime import datetime
import sys

# import project modules
from src.churn_prediction.logger import logger
from src.churn_prediction.pydantic.pipeline_config import PipelineConfig
from src.churn_prediction.components.data_ingestion import DataIngester
from src.churn_prediction.components.data_validation import DataValidator
from src.churn_prediction.components.data_transformation import DataTransformer
from src.churn_prediction.utils.common import load_single_config, get_execution_date

25/11/14 18:22:44 WARN Utils: Your hostname, Supawits-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.4 instead (on interface en0)
25/11/14 18:22:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/14 18:22:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 57455)
Traceback (most recent call last):
  File "/Users/supawitjunsiritrakhoon/Desktop/Customer_Churn_Prediction/Project_file/customer-churn-prediction/venv/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Users/supawitjunsiritrakhoon/Desktop/Customer_Churn_Prediction/Project_file/customer-churn-prediction/venv/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/Users/supawitjunsiritrakhoon/Desktop/Customer_Churn_Prediction/Project_file/customer-churn-prediction/venv/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Users/supawitjunsiritrakhoon/Desktop/Customer_Churn_Prediction/Project_file/customer-churn-prediction/venv/lib/python3.12/socketserver.py", l

In [ ]:
def execute(
    config_path: str,
    execution_date: Optional[str] = None,
) -> None:
    """
    Execute the data and ml pipeline stages base on configuration.

    Args:
        config_path (str): Path to the configuration file.
        execution_date (Optional[str]): Execution date for the pipeline run.

    Returns:
        None
    """
    # Load config
    config_path: Path = Path(config_path)
    pipeline_config: PipelineConfig = load_single_config(PipelineConfig, config_path)
    execution_date: str = get_execution_date(execution_date) if execution_date else datetime.now().strftime("%Y-%m-%d")

    pipeline_process = {
        "ingestion": DataIngester,
        "validation": DataValidator,
        "transformation": DataTransformer,
    }

    for process_name, process_class in pipeline_process.items():
        process_method = getattr(pipeline_config, process_name, None)
        if not process_method:
            continue
        process_class(config_path, execution_date).run()

    logger.info("Pipeline execution completed.")

In [ ]:
def main():
    """
    Main function to execute the pipeline with command-line arguments.

    Command-line Arguments:
        1. config_path (str): Path to the configuration file.
        2. execution_date (Optional[str]): Execution date for the pipeline run.

    Example:
        python main.py config/config.yaml 2024-01-01

    This function retrieves command-line arguments and calls the execute function.
    """
    try:
        config_path = sys.argv[1] if len(sys.argv) > 1 else None
        execution_date = sys.argv[2] if len(sys.argv) > 2 else None

        execute(
            config_path=config_path,
            execution_date=execution_date,
        )
    except Exception as e:
        raise e